In [37]:
# ============================================================
# 电商用户数据清洗与预处理
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')  # 忽略警告

# 设置显示选项
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

print("=" * 60)
print("开始数据清洗...")
print("=" * 60)

开始数据清洗...


In [38]:
# ============================================================
# 1. 读取数据
# ============================================================

from pathlib import Path
import pandas as pd

def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "E Commerce Dataset.xlsx").exists():
            return candidate
    raise FileNotFoundError("未找到项目根目录")

ROOT = find_workspace_root()

print("\n【步骤1】读取数据...")

file_path = ROOT / "data" / "E Commerce Dataset.xlsx"
df = pd.read_excel(file_path, sheet_name="E Comm")

print(f"✅ 读取成功！")
print(f"✅ 数据形状：{df.shape[0]} 行，{df.shape[1]} 列")
print("\n前5行数据预览：")
print(df.head())


【步骤1】读取数据...
✅ 读取成功！
✅ 数据形状：5630 行，20 列

前5行数据预览：
   CustomerID  Churn  Tenure PreferredLoginDevice  CityTier  WarehouseToHome  \
0       50001      1    4.00         Mobile Phone         3             6.00   
1       50002      1     NaN                Phone         1             8.00   
2       50003      1     NaN                Phone         1            30.00   
3       50004      1    0.00                Phone         3            15.00   
4       50005      1    0.00                Phone         1            12.00   

  PreferredPaymentMode  Gender  HourSpendOnApp  NumberOfDeviceRegistered  \
0           Debit Card  Female            3.00                         3   
1                  UPI    Male            3.00                         4   
2           Debit Card    Male            2.00                         4   
3           Debit Card    Male            2.00                         4   
4                   CC    Male             NaN                         3   

     Prefer

In [39]:
# ============================================================
# 2. 理解数据
# ============================================================

print("\n" + "=" * 60)
print("【步骤2】理解数据")
print("=" * 60)

print("\n数据信息：")
df.info()

print("\n" + "-" * 40)
print("📝 答案：")
print("-" * 40)
print("1. 一行数据代表：一个用户（客户）的电商行为记录")
print("2. 用户唯一标识字段：CustomerID")
print("3. 流失分析的目标变量：Churn（1=已流失，0=未流失）")


【步骤2】理解数据

数据信息：
<class 'pandas.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   str    
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   str    
 7   Gender                       5630 non-null   str    
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   str    
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   str    
 13  NumberOfAdd

In [40]:
# ============================================================
# 3. 缺失值报告
# ============================================================

print("\n" + "=" * 60)
print("【步骤3】缺失值报告")
print("=" * 60)

missing_count = df.isna().sum()
missing_ratio = (df.isna().mean() * 100).round(2)

missing_report = pd.DataFrame({
    "缺失数量": missing_count,
    "缺失比例(%)": missing_ratio
}).sort_values("缺失数量", ascending=False)

print(missing_report)


【步骤3】缺失值报告
                             缺失数量  缺失比例(%)
DaySinceLastOrder             307     5.45
OrderAmountHikeFromlastYear   265     4.71
Tenure                        264     4.69
OrderCount                    258     4.58
CouponUsed                    256     4.55
HourSpendOnApp                255     4.53
WarehouseToHome               251     4.46
CustomerID                      0     0.00
PreferredLoginDevice            0     0.00
Churn                           0     0.00
PreferredPaymentMode            0     0.00
CityTier                        0     0.00
SatisfactionScore               0     0.00
PreferedOrderCat                0     0.00
NumberOfDeviceRegistered        0     0.00
Gender                          0     0.00
Complain                        0     0.00
NumberOfAddress                 0     0.00
MaritalStatus                   0     0.00
CashbackAmount                  0     0.00


In [41]:
# ============================================================
# 4. 重复记录检查
# ============================================================

print("\n" + "=" * 60)
print("【步骤4】重复记录检查")
print("=" * 60)

duplicate_rows = df.duplicated().sum()
duplicate_customer_ids = df["CustomerID"].duplicated().sum()

print(f"完全重复行数：{duplicate_rows}")
print(f"CustomerID 重复数量：{duplicate_customer_ids}")
print("\n💡 注意：CustomerID 重复不能直接删除")
print("   原因：同一用户可能有多次购买记录，每条记录代表不同交易")


【步骤4】重复记录检查
完全重复行数：0
CustomerID 重复数量：0

💡 注意：CustomerID 重复不能直接删除
   原因：同一用户可能有多次购买记录，每条记录代表不同交易


In [42]:
# ============================================================
# 5. 缺失值处理（用中位数填充）
# ============================================================

print("\n" + "=" * 60)
print("【步骤5】缺失值处理（中位数填充）")
print("=" * 60)

# ✅ 修正：列名用 OrderAmountHikeFromlastYear（小写 l）
numeric_missing_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",  # ← 这里是关键！小写 l
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

for col in numeric_missing_cols:
    if col in df.columns:
        median_val = df[col].median()
        df.loc[:, col] = df[col].fillna(median_val)
        print(f"✅ {col}：用中位数 {median_val:.2f} 填充")
    else:
        print(f"⚠️ 列 '{col}' 不存在，跳过")

# 验证是否还有缺失
remaining = df[numeric_missing_cols].isna().sum()
print(f"\n剩余缺失值统计：")
print(remaining)


【步骤5】缺失值处理（中位数填充）
✅ Tenure：用中位数 9.00 填充
✅ WarehouseToHome：用中位数 14.00 填充
✅ HourSpendOnApp：用中位数 3.00 填充
✅ OrderAmountHikeFromlastYear：用中位数 15.00 填充
✅ CouponUsed：用中位数 1.00 填充
✅ OrderCount：用中位数 2.00 填充
✅ DaySinceLastOrder：用中位数 3.00 填充

剩余缺失值统计：
Tenure                         0
WarehouseToHome                0
HourSpendOnApp                 0
OrderAmountHikeFromlastYear    0
CouponUsed                     0
OrderCount                     0
DaySinceLastOrder              0
dtype: int64


In [43]:
# ============================================================
# 6. 类别字段标准化
# ============================================================

print("\n" + "=" * 60)
print("【步骤6】类别字段标准化")
print("=" * 60)

category_cols = [
    "PreferredLoginDevice",
    "PreferredPaymentMode",
    "PreferedOrderCat",
]

# 查看原始分布
print("\n📋 原始类别分布：")
for col in category_cols:
    if col in df.columns:
        print(f"\n{col}：")
        print(df[col].value_counts())

# 执行标准化
print("\n🔄 执行标准化...")

if "PreferredLoginDevice" in df.columns:
    df.loc[:, "PreferredLoginDevice"] = df["PreferredLoginDevice"].replace({
        "Phone": "Mobile Phone"
    })

if "PreferredPaymentMode" in df.columns:
    df.loc[:, "PreferredPaymentMode"] = df["PreferredPaymentMode"].replace({
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    })

if "PreferedOrderCat" in df.columns:
    df.loc[:, "PreferedOrderCat"] = df["PreferedOrderCat"].replace({
        "Mobile": "Mobile Phone"
    })

# 查看标准化后分布
print("\n✅ 标准化后的类别分布：")
for col in category_cols:
    if col in df.columns:
        print(f"\n{col}：")
        print(df[col].value_counts())


【步骤6】类别字段标准化

📋 原始类别分布：

PreferredLoginDevice：
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode：
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat：
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64

🔄 执行标准化...

✅ 标准化后的类别分布：

PreferredLoginDevice：
PreferredLoginDevice
Mobile Phone    3996
Computer        1634
Name: count, dtype: int64

PreferredPaymentMode：
PreferredPaymentMode
Debit Card          2314
Credit Card         1774
E wallet             614
Cash on Delivery     514
UPI                  414
Name: count, dtype: int64

PreferedOrderCat：
PreferedOrderCat
Mobi

In [44]:
# ============================================================
# 7. 候选异常值检查（IQR方法）
# ============================================================

print("\n" + "=" * 60)
print("【步骤7】候选异常值检查（IQR方法）")
print("=" * 60)


def iqr_outlier_summary(series):
    """返回数值字段的 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return pd.Series({
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": ((series < lower) | (series > upper)).sum()
    })


# 检查三个字段
check_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
for col in check_cols:
    if col in df.columns:
        print(f"\n📊 {col}：")
        print(iqr_outlier_summary(df[col]))

print("\n💡 重要提示：候选异常值不能直接删除！")
print("   1. IQR只是统计标记，不代表数据错误")
print("   2. 可能代表真实的业务情况（如大客户高消费）")
print("   3. 应结合业务背景进一步验证")


【步骤7】候选异常值检查（IQR方法）

📊 WarehouseToHome：
Q1         9.00
Q3        20.00
下限        -7.50
上限        36.50
候选异常值数量    2.00
dtype: float64

📊 OrderCount：
Q1          1.00
Q3          3.00
下限         -2.00
上限          6.00
候选异常值数量   703.00
dtype: float64

📊 CashbackAmount：
Q1        145.77
Q3        196.39
下限         69.84
上限        272.33
候选异常值数量   438.00
dtype: float64

💡 重要提示：候选异常值不能直接删除！
   1. IQR只是统计标记，不代表数据错误
   2. 可能代表真实的业务情况（如大客户高消费）
   3. 应结合业务背景进一步验证


In [45]:
# ============================================================
# 8. 业务规则检查
# ============================================================

print("\n" + "=" * 60)
print("【步骤8】业务规则检查")
print("=" * 60)

rules = {}
if "Tenure" in df.columns:
    rules["使用时长小于 0"] = (df["Tenure"] < 0).sum()
if "WarehouseToHome" in df.columns:
    rules["仓库距离小于 0"] = (df["WarehouseToHome"] < 0).sum()
if "OrderCount" in df.columns:
    rules["订单数小于或等于 0"] = (df["OrderCount"] <= 0).sum()
if "CashbackAmount" in df.columns:
    rules["返现金额小于 0"] = (df["CashbackAmount"] < 0).sum()

rule_result = pd.Series(rules)
print("不符合业务规则的记录数量：")
print(rule_result)


【步骤8】业务规则检查
不符合业务规则的记录数量：
使用时长小于 0      0
仓库距离小于 0      0
订单数小于或等于 0    0
返现金额小于 0      0
dtype: int64


In [46]:
# ============================================================
# 9. 清洗验收
# ============================================================

print("\n" + "=" * 60)
print("【步骤9】清洗验收")
print("=" * 60)

try:
    # 检查数值字段是否还有缺失
    missing_check = 0
    for col in numeric_missing_cols:
        if col in df.columns:
            missing_check += df[col].isna().sum()
    assert missing_check == 0, "❌ 数值字段仍有缺失值"

    if "PreferredLoginDevice" in df.columns:
        assert "Phone" not in df["PreferredLoginDevice"].unique(), "❌ 登录设备尚未统一"
    if "PreferredPaymentMode" in df.columns:
        assert "COD" not in df["PreferredPaymentMode"].unique(), "❌ 支付方式尚未统一"
        assert "CC" not in df["PreferredPaymentMode"].unique(), "❌ 支付方式尚未统一"

    print("✅ 所有验收通过！数据清洗成功！")
except AssertionError as e:
    print(f"❌ 验收失败：{e}")


【步骤9】清洗验收
✅ 所有验收通过！数据清洗成功！


In [47]:
# ============================================================
# 10. 导出清洗后的数据
# ============================================================

print("\n" + "=" * 60)
print("【步骤10】导出数据")
print("=" * 60)

OUTPUT_PATH = Path("../output/day04_project/ecommerce_customer_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"✅ 已导出到：{OUTPUT_PATH.resolve()}")
print(f"✅ 导出数据共 {df.shape[0]} 行，{df.shape[1]} 列")


【步骤10】导出数据
✅ 已导出到：C:\Users\21975\PyCharmMiscProject\output\day04_project\ecommerce_customer_cleaned.csv
✅ 导出数据共 5630 行，20 列


In [48]:
# ============================================================
# 11. 数据统计摘要
# ============================================================

print("\n" + "=" * 60)
print("【数据统计摘要】")
print("=" * 60)

print(f"\n📊 数据概况：")
print(f"   - 总用户数：{df['CustomerID'].nunique()}")
print(f"   - 总记录数：{len(df)}")
print(f"   - 字段数：{len(df.columns)}")
if "Churn" in df.columns:
    print(f"   - 流失用户数：{df['Churn'].sum()}")
    print(f"   - 流失率：{df['Churn'].mean() * 100:.2f}%")


【数据统计摘要】

📊 数据概况：
   - 总用户数：5630
   - 总记录数：5630
   - 字段数：20
   - 流失用户数：948
   - 流失率：16.84%


In [49]:
# ============================================================
# 12. 课后思考
# ============================================================

print("\n" + "=" * 60)
print("【课后思考】")
print("=" * 60)
print("""
📌 可用于预测流失的特征：
   ✓ Tenure（使用时长）
   ✓ WarehouseToHome（仓库距离）
   ✓ HourSpendOnApp（App使用时间）
   ✓ OrderCount（订单数量）
   ✓ DaySinceLastOrder（距上次下单天数）
   ✓ SatisfactionScore（满意度评分）
   ✓ PreferredLoginDevice（登录设备）
   ✓ PreferredPaymentMode（支付方式）

❌ CustomerID 不应该用于建模，因为：
   1. 它是唯一标识符，不具备预测能力
   2. 可能导致模型过拟合
   3. 训练集和测试集中没有对应关系
   4. 实际应用中无法对新用户进行预测
""")

print("\n" + "=" * 60)
print("🎉 全部完成！")
print("=" * 60)


【课后思考】

📌 可用于预测流失的特征：
   ✓ Tenure（使用时长）
   ✓ WarehouseToHome（仓库距离）
   ✓ HourSpendOnApp（App使用时间）
   ✓ OrderCount（订单数量）
   ✓ DaySinceLastOrder（距上次下单天数）
   ✓ SatisfactionScore（满意度评分）
   ✓ PreferredLoginDevice（登录设备）
   ✓ PreferredPaymentMode（支付方式）

❌ CustomerID 不应该用于建模，因为：
   1. 它是唯一标识符，不具备预测能力
   2. 可能导致模型过拟合
   3. 训练集和测试集中没有对应关系
   4. 实际应用中无法对新用户进行预测


🎉 全部完成！


In [50]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

In [51]:
from pathlib import Path
import pandas as pd

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start] + list(start.parents):
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("找不到项目根目录（包含 data 文件夹的目录）")

ROOT = find_project_root()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "output" / "day04_project"

print("项目根目录：", ROOT)
print("数据目录：", DATA_DIR)
print("输出目录：", OUTPUT_DIR)

file_path = DATA_DIR / "E Commerce Dataset.xlsx"
raw_df = pd.read_excel(file_path, sheet_name="E Comm")

print(f"✅ 数据读取成功！形状：{raw_df.shape}")
print(raw_df.head())

项目根目录： C:\Users\21975\PyCharmMiscProject
数据目录： C:\Users\21975\PyCharmMiscProject\data
输出目录： C:\Users\21975\PyCharmMiscProject\output\day04_project
✅ 数据读取成功！形状：(5630, 20)
   CustomerID  Churn  Tenure PreferredLoginDevice  CityTier  WarehouseToHome  \
0       50001      1    4.00         Mobile Phone         3             6.00   
1       50002      1     NaN                Phone         1             8.00   
2       50003      1     NaN                Phone         1            30.00   
3       50004      1    0.00                Phone         3            15.00   
4       50005      1    0.00                Phone         1            12.00   

  PreferredPaymentMode  Gender  HourSpendOnApp  NumberOfDeviceRegistered  \
0           Debit Card  Female            3.00                         3   
1                  UPI    Male            3.00                         4   
2           Debit Card    Male            2.00                         4   
3           Debit Card    Male            2.0

In [52]:
# ==================== 2. 构建数据质量报告 ====================

def build_quality_report(data):
    report = pd.DataFrame({
        "数据类型": data.dtypes,
        "缺失数量": data.isna().sum(),
        "缺失比例(%)": (data.isna().sum() / len(data) * 100).round(2),
        "唯一值数量": data.nunique()
    })
    return report

quality_before = build_quality_report(raw_df)
print("\n=== 清洗前数据质量报告 ====")
print(quality_before)


=== 清洗前数据质量报告 ====
                                数据类型  缺失数量  缺失比例(%)  唯一值数量
CustomerID                     int64     0     0.00   5630
Churn                          int64     0     0.00      2
Tenure                       float64   264     4.69     36
PreferredLoginDevice             str     0     0.00      3
CityTier                       int64     0     0.00      3
WarehouseToHome              float64   251     4.46     34
PreferredPaymentMode             str     0     0.00      7
Gender                           str     0     0.00      2
HourSpendOnApp               float64   255     4.53      6
NumberOfDeviceRegistered       int64     0     0.00      6
PreferedOrderCat                 str     0     0.00      6
SatisfactionScore              int64     0     0.00      5
MaritalStatus                    str     0     0.00      3
NumberOfAddress                int64     0     0.00     15
Complain                       int64     0     0.00      2
OrderAmountHikeFromlastYear  float64

In [53]:
# ==================== 3. 初始审计 ====================

print(f"\n完全重复行数：{raw_df.duplicated().sum()}")
print(f"CustomerID 重复数量：{raw_df['CustomerID'].duplicated().sum()}")
print("\nChurn 分布：")
print(raw_df["Churn"].value_counts())
print(f"\n流失率：{raw_df['Churn'].mean():.2%}")

for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts())



完全重复行数：0
CustomerID 重复数量：0

Churn 分布：
Churn
0    4682
1     948
Name: count, dtype: int64

流失率：16.84%

PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


In [54]:
# ==================== 4. 定义清洗规则 ====================

NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {"Phone": "Mobile Phone"},
    "PreferredPaymentMode": {"COD": "Cash on Delivery", "CC": "Credit Card"},
    "PreferedOrderCat": {"Mobile": "Mobile Phone"}
}

In [55]:
# ==================== 5. 清洗函数 ====================

def clean_ecommerce_data(data):
    df = data.copy()
    logs = []
    initial_rows = len(df)

    # 删除完全重复行
    duplicate_count = df.duplicated().sum()
    if duplicate_count > 0:
        df = df.drop_duplicates()
        logs.append({
            "处理步骤": "删除完全重复行",
            "处理规则": "删除完全相同的记录",
            "处理前记录数": initial_rows,
            "处理后记录数": len(df),
            "影响记录数": duplicate_count
        })

    # 数值字段中位数填补
    for col in NUMERIC_MISSING_COLS:
        if col in df.columns:
            missing_count = df[col].isna().sum()
            if missing_count > 0:
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                logs.append({
                    "处理步骤": f"填补缺失值 - {col}",
                    "处理规则": f"使用中位数 {median_val:.2f} 填补",
                    "处理前记录数": len(df),
                    "处理后记录数": len(df),
                    "影响记录数": missing_count
                })

    # 类别标准化
    for col, mapping in CATEGORY_MAPPINGS.items():
        if col in df.columns:
            for old_val, new_val in mapping.items():
                affected_count = (df[col] == old_val).sum()
                if affected_count > 0:
                    df[col] = df[col].replace(old_val, new_val)
                    logs.append({
                        "处理步骤": f"类别标准化 - {col}",
                        "处理规则": f"'{old_val}' -> '{new_val}'",
                        "处理前记录数": len(df),
                        "处理后记录数": len(df),
                        "影响记录数": affected_count
                    })

    # 类型转换
    if "Churn" in df.columns:
        df["Churn"] = df["Churn"].astype(int)
    if "Complain" in df.columns:
        df["Complain"] = df["Complain"].astype(int)

    cleaning_log = pd.DataFrame(logs)
    return df, cleaning_log


cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)
print("\n=== 清洗日志 ===")
print(cleaning_log)


=== 清洗日志 ===
                                   处理步骤                         处理规则  处理前记录数  \
0                        填补缺失值 - Tenure                使用中位数 9.00 填补    5630   
1               填补缺失值 - WarehouseToHome               使用中位数 14.00 填补    5630   
2                填补缺失值 - HourSpendOnApp                使用中位数 3.00 填补    5630   
3   填补缺失值 - OrderAmountHikeFromlastYear               使用中位数 15.00 填补    5630   
4                    填补缺失值 - CouponUsed                使用中位数 1.00 填补    5630   
5                    填补缺失值 - OrderCount                使用中位数 2.00 填补    5630   
6             填补缺失值 - DaySinceLastOrder                使用中位数 3.00 填补    5630   
7          类别标准化 - PreferredLoginDevice    'Phone' -> 'Mobile Phone'    5630   
8          类别标准化 - PreferredPaymentMode  'COD' -> 'Cash on Delivery'    5630   
9          类别标准化 - PreferredPaymentMode        'CC' -> 'Credit Card'    5630   
10             类别标准化 - PreferedOrderCat   'Mobile' -> 'Mobile Phone'    5630   

    处理后记录数  影响记录数  
0    

In [56]:
# ==================== 6. 特征工程 ====================

def iqr_outlier_summary(series):
    series = series.dropna()
    if len(series) == 0:
        return {"Q1": np.nan, "Q3": np.nan, "下限": np.nan, "上限": np.nan, "候选异常值数量": 0}
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }


# 新增特征
tenure_bins = [0, 12, 24, 36, 48, 100]
tenure_labels = ["0-1年", "1-2年", "2-3年", "3-4年", "4年以上"]
cleaned_df["TenureGroup"] = pd.cut(cleaned_df["Tenure"], bins=tenure_bins, labels=tenure_labels, right=False)
cleaned_df["IsMobileLogin"] = (cleaned_df["PreferredLoginDevice"] == "Mobile Phone").astype(int)

# 异常值报告
outlier_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_report = pd.DataFrame({"字段": outlier_cols})
for col in outlier_cols:
    summary = iqr_outlier_summary(cleaned_df[col])
    for key, val in summary.items():
        outlier_report.loc[outlier_report["字段"] == col, key] = val

print("\n=== 候选异常值报告 ===")
print(outlier_report)


=== 候选异常值报告 ===
                字段     Q1     Q3    下限     上限  候选异常值数量
0  WarehouseToHome   9.00  20.00 -7.50  36.50     2.00
1       OrderCount   1.00   3.00 -2.00   6.00   703.00
2   CashbackAmount 145.77 196.39 69.84 272.33   438.00


In [57]:
# ==================== 7. 业务规则检查 ====================

business_rule_report = pd.DataFrame({
    "规则": ["使用时长小于 0", "仓库距离小于 0", "订单数小于或等于 0", "返现金额小于 0"],
    "不合规记录数": [
        (cleaned_df["Tenure"] < 0).sum(),
        (cleaned_df["WarehouseToHome"] < 0).sum(),
        (cleaned_df["OrderCount"] <= 0).sum(),
        (cleaned_df["CashbackAmount"] < 0).sum()
    ]
})
print("\n=== 业务规则检查 ===")
print(business_rule_report)


=== 业务规则检查 ===
           规则  不合规记录数
0    使用时长小于 0       0
1    仓库距离小于 0       0
2  订单数小于或等于 0       0
3    返现金额小于 0       0


In [59]:
# ==================== 8. 验收与导出 ====================

quality_after = build_quality_report(cleaned_df)

# 验证
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0, "数值字段仍有缺失值"
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique(), "登录设备未完全标准化"
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique(), "支付方式未完全标准化"
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique(), "支付方式未完全标准化"
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns), "新增特征缺失"

print("\n✅ 所有验证通过！")

# 导出文件
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=False, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=False, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False, encoding="utf-8-sig")
business_rule_report.to_csv(OUTPUT_DIR / "business_rule_report.csv", index=False, encoding="utf-8-sig")

print(f"\n✅ 所有文件已导出到：{OUTPUT_DIR}")
print("1. ecommerce_customer_cleaned.csv - 清洗后的用户数据")
print("2. data_quality_before.csv - 清洗前质量报告")
print("3. data_quality_after.csv - 清洗后质量报告")
print("4. cleaning_log.csv - 数据处理日志")
print("5. outlier_report.csv - 候选异常值报告")
print("6. business_rule_report.csv - 业务规则检查报告")


✅ 所有验证通过！

✅ 所有文件已导出到：C:\Users\21975\PyCharmMiscProject\output\day04_project
1. ecommerce_customer_cleaned.csv - 清洗后的用户数据
2. data_quality_before.csv - 清洗前质量报告
3. data_quality_after.csv - 清洗后质量报告
4. cleaning_log.csv - 数据处理日志
5. outlier_report.csv - 候选异常值报告
6. business_rule_report.csv - 业务规则检查报告
